# Objective 2 — Power BI Dashboard Preparation
**AI Trust Paradox Among Software Developers**  
AICW Fellowship 2026 | Edunet Foundation | Galgotias University  

**What this notebook does:**  
1. Loads the cleaned dataset (trust_paradox_clean.csv) from Objective 1  
2. Computes every statistic used in all 4 Power BI dashboard pages  
3. Creates the Power BI optimised CSV (37 columns instead of 187)  
4. Exports 4 analysis summary CSVs (one per dashboard page)  
5. Auto-downloads all files to your computer  

**Input :** trust_paradox_clean.csv (output from Objective 1)  
**Outputs:** trust_paradox_powerbi.csv + 4 analysis CSVs  

---
Run cells one by one in order. Each cell prints its results before you move on.

## Cell 1 — Install and import libraries

In [ ]:
# All libraries below come pre-installed in Google Colab — no pip install needed
import pandas as pd
import numpy as np
import warnings
import os
from scipy import stats  # for chi-square test

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: f'{x:.3f}')
pd.set_option('display.max_columns', 50)

print('Libraries loaded:')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')
print(f'  scipy   : {stats.__version__}')
print()
print('Run Cell 2 to upload trust_paradox_clean.csv')

## Cell 2 — Upload trust_paradox_clean.csv
> **Important:** This is the file produced by Objective 1 (your cleaned dataset).  
> File size is ~128 MB. Upload takes 1-3 minutes. Wait for 100% before Cell 3.

In [ ]:
from google.colab import files

print('File chooser will appear below.')
print('Select: trust_paradox_clean.csv  (~128 MB)')
print('Wait for upload to reach 100% before running Cell 3.')
print()

uploaded = files.upload()
filename  = list(uploaded.keys())[0]
INPUT_PATH = f'/content/{filename}'

print(f'\nUploaded : {filename}')
print(f'Size     : {len(uploaded[filename])/1024/1024:.1f} MB')
print(f'Path     : {INPUT_PATH}')

## Cell 3 — Load and verify the cleaned dataset

In [ ]:
print('Loading dataset...')
df = pd.read_csv(INPUT_PATH, low_memory=False)

print(f'\nRows    : {len(df):,}')
print(f'Columns : {len(df.columns)}')

# Verify all required columns exist
REQUIRED = [
    'AIAcc_clean', 'AIAcc_score', 'AISent_clean', 'AISent_score',
    'AIThreat_clean', 'JobSat_clean', 'UsesAI', 'ExperienceBand',
    'ToolGroup', 'YearsCode_num',
    'Frust_AlmostRight', 'Frust_Debugging', 'Frust_LessConfident',
]
missing_cols = [c for c in REQUIRED if c not in df.columns]

if missing_cols:
    print(f'\nMISSING COLUMNS: {missing_cols}')
    print('Make sure you uploaded the output of Objective 1, not the raw CSV.')
else:
    print('All required derived columns present ✓')

# Quick sanity checks
assert len(df) == 48195, f'Expected 48,195 rows, got {len(df):,}'
print(f'Row count verified: 48,195 ✓')

# Show key column dtypes
print('\nKey column types:')
for col in ['AIAcc_score','AISent_score','YearsCode_num','JobSat']:
    if col in df.columns:
        print(f'  {col:<20} {df[col].dtype}')

---
# PART A — Paradox 1: Use-Despite-Distrust
**Variables used:** AIAcc_clean, AIAcc_score, UsesAI, ExperienceBand, Country, DevType

## Cell 4 — P1: Trust level distribution (headline KPI)

In [ ]:
# Trust level distribution — excludes No Response from denominator
trust_ans = df[df['AIAcc_clean'] != 'No Response'].copy()
n_trust   = len(trust_ans)

# Adoption rate
uses_ans   = df[df['UsesAI'] != 'No Response']
adopt_rate = (uses_ans['UsesAI'] == 'Uses AI').mean() * 100

# Trust distribution
TRUST_ORDER = [
    'Highly trust', 'Somewhat trust',
    'Neither trust nor distrust',
    'Somewhat distrust', 'Highly distrust'
]

print('=' * 55)
print('  PARADOX 1 — KEY PERFORMANCE INDICATORS')
print('=' * 55)
print(f'\n  Adoption rate        : {adopt_rate:.1f}%  (target: 78.5%)')
print(f'  Answered trust Q     : {n_trust:,}')
print(f'  No Response (trust Q): {(df["AIAcc_clean"]=="No Response").sum():,}  (MNAR — retained)\n')

print(f'  {"Trust Level":<35} {"Count":>7} {"Percent":>8}')
print(f'  {"-"*52}')

p1_dist = []
for level in TRUST_ORDER:
    cnt = int((trust_ans['AIAcc_clean'] == level).sum())
    pct = cnt / n_trust * 100
    bar = '█' * int(pct / 2)
    print(f'  {level:<35} {cnt:>7,}  {pct:>6.1f}%  {bar}')
    p1_dist.append({'Trust_Level': level, 'Count': cnt, 'Percent': round(pct, 1)})

# Aggregate metrics
ht_pct  = (trust_ans['AIAcc_clean'] == 'Highly trust').mean()  * 100
ad_pct  = trust_ans['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean() * 100
mean_sc = df['AIAcc_score'].mean()

print(f'\n  HEADLINE NUMBERS FOR POWER BI CARDS:')
print(f'  Adoption Rate         = {adopt_rate:.1f}%')
print(f'  High Trust Rate       = {ht_pct:.1f}%')
print(f'  Any Distrust Rate     = {ad_pct:.1f}%')
print(f'  Avg Trust Score       = {mean_sc:.3f}  (below 3.0 = distrust direction)')
print(f'  Paradox Severity      = {adopt_rate - ht_pct:.1f} pts')

# Save for export
p1_dist_df = pd.DataFrame(p1_dist)
print('\n[P1 distribution saved for export]')

## Cell 5 — P1: Cross-tabulation UsesAI × AIAcc_clean (the core Paradox 1 chart)

In [ ]:
# This is the stacked bar chart in Power BI Page 2
# Shows: among active AI users, what % distrust vs trust?

# Filter: only rows where BOTH UsesAI and AIAcc_clean are answered
both_ans = df[
    (df['UsesAI'] != 'No Response') &
    (df['AIAcc_clean'] != 'No Response')
].copy()

print('Cross-tabulation: UsesAI × AIAcc_clean')
print('(% of row — each row sums to 100%)')
print()

crosstab_rows = []
for group in ['Uses AI', 'Does Not Use AI']:
    grp = both_ans[both_ans['UsesAI'] == group]
    n   = len(grp)
    row = {'UsesAI_Group': group, 'n': n}
    print(f'{group} (n={n:,}):')
    for level in TRUST_ORDER:
        cnt = int((grp['AIAcc_clean'] == level).sum())
        pct = cnt / n * 100
        row[level] = round(pct, 1)
        bar = '█' * int(pct / 3)
        print(f'  {level:<35} {pct:>5.1f}%  {bar}')
    any_dist = grp['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean() * 100
    row['Any_Distrust_Pct'] = round(any_dist, 1)
    print(f'  → ANY DISTRUST: {any_dist:.1f}%')
    print()
    crosstab_rows.append(row)

# Chi-square test
ct = pd.crosstab(both_ans['UsesAI'], both_ans['AIAcc_clean'])
chi2, p_val, dof, expected = stats.chi2_contingency(ct)
n_ct   = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n_ct * (min(ct.shape) - 1)))

print(f'Chi-square test of independence:')
print(f'  χ² = {chi2:,.2f}')
print(f'  p  = {p_val:.4f}  ({"SIGNIFICANT p<0.0001" if p_val < 0.0001 else "not significant"})')
print(f'  df = {dof}')
print(f'  Cramér\'s V = {cramers_v:.3f}  (>0.3 = LARGE effect)')

p1_crosstab_df = pd.DataFrame(crosstab_rows)
print('\n[P1 cross-tab saved for export]')

## Cell 6 — P1: Expertise effect (trust by ExperienceBand)

In [ ]:
# The most counterintuitive finding: trust FALLS as experience RISES

print('Trust by ExperienceBand — the expertise effect')
print()
print(f'{"Band":<30} {"n":>7} {"High Trust %":>13} {"Any Distrust %":>15}')
print('-' * 68)

BAND_ORDER = [
    'Early Career (0-2 yrs)',
    'Mid-Early (3-5 yrs)',
    'Mid (6-10 yrs)',
    'Experienced (10+ yrs)'
]

exp_rows = []
for band in BAND_ORDER:
    g   = trust_ans[trust_ans['ExperienceBand'] == band]
    n   = len(g)
    ht  = (g['AIAcc_clean'] == 'Highly trust').mean()  * 100
    ad  = g['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean() * 100
    avg = g['AIAcc_score'].mean()
    print(f'{band:<30} {n:>7,} {ht:>12.1f}% {ad:>14.1f}%')
    exp_rows.append({
        'ExperienceBand': band, 'n': n,
        'High_Trust_Pct': round(ht,1),
        'Any_Distrust_Pct': round(ad,1),
        'Avg_Trust_Score': round(avg,3)
    })

print()
print('Key insight: Early Career 10.0% trust → Experienced 2.3% trust')
print('Trust FALLS monotonically at every experience level.')
print('Experienced developers also show HIGHEST distrust (49.8%)')

p1_expertise_df = pd.DataFrame(exp_rows)
print('\n[P1 expertise table saved for export]')

## Cell 7 — P1: Country trust analysis

In [ ]:
# Country-level trust analysis
# Only countries with at least 200 respondents who answered the trust question

country_rows = []
country_data = trust_ans.groupby('Country').apply(
    lambda g: pd.Series({
        'n': len(g),
        'High_Trust_Pct': round((g['AIAcc_clean']=='Highly trust').mean()*100, 1),
        'Any_Distrust_Pct': round(g['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean()*100, 1),
        'Avg_Score': round(g['AIAcc_score'].mean(), 3)
    })
).reset_index()

# Only countries with n >= 200
country_data = country_data[country_data['n'] >= 200].sort_values('High_Trust_Pct', ascending=False)

print(f'Countries with 200+ respondents: {len(country_data)}')
print()
print(f'{"Country":<45} {"n":>6} {"High Trust":>11} {"Any Distrust":>13}')
print('-' * 78)
for _, row in country_data.head(15).iterrows():
    print(f'{str(row["Country"])[:44]:<45} {int(row["n"]):>6,} {row["High_Trust_Pct"]:>10.1f}% {row["Any_Distrust_Pct"]:>12.1f}%')

print()
top    = country_data.iloc[0]
bottom = country_data.iloc[-1]
print(f'Highest trust: {top["Country"]} ({top["High_Trust_Pct"]}%)')
print(f'Lowest trust:  {bottom["Country"]} ({bottom["High_Trust_Pct"]}%)')

p1_country_df = country_data.copy()
print('\n[P1 country data saved for export]')

---
# PART B — Paradox 2: Adoption-Ethics Gap
**Variables used:** AISent_clean, AISent_score, AIAcc_clean, ToolGroup, Frust_ columns

## Cell 8 — P2: Ethics gap (sentiment vs trust)

In [ ]:
# The ethics gap = positive sentiment rate MINUS positive trust rate

sent_ans = df[df['AISent_clean'].isin([
    'Very favorable','Favorable','Indifferent','Unfavorable','Very unfavorable'
])].copy()

pos_sent  = sent_ans['AISent_clean'].isin(['Very favorable','Favorable']).mean() * 100
pos_trust = trust_ans['AIAcc_clean'].isin(['Highly trust','Somewhat trust']).mean()  * 100
gap       = pos_sent - pos_trust

mean_sent  = df['AISent_score'].mean()
mean_trust = df['AIAcc_score'].mean()

print('=' * 55)
print('  PARADOX 2 — ETHICS GAP ANALYSIS')
print('=' * 55)
print(f'\n  Positive Sentiment (Very/Favorable) : {pos_sent:.1f}%  (n={sent_ans["AISent_clean"].isin(["Very favorable","Favorable"]).sum():,})')
print(f'  Positive Trust  (Highly/Somewhat)   : {pos_trust:.1f}%  (n={trust_ans["AIAcc_clean"].isin(["Highly trust","Somewhat trust"]).sum():,})')
print(f'  ETHICS GAP                          : {gap:.1f} percentage points')
print(f'\n  Mean AISent_score : {mean_sent:.3f}  (above 3.0 = positive direction)')
print(f'  Mean AIAcc_score  : {mean_trust:.3f}  (below 3.0 = distrust direction)')
print(f'  Score gap         : {mean_sent - mean_trust:.3f} pts on 1-5 scale')

print(f'\nSentiment distribution (n={len(sent_ans):,}):')
SENT_ORDER = ['Very favorable','Favorable','Indifferent','Unfavorable','Very unfavorable']
p2_sent_rows = []
for level in SENT_ORDER:
    cnt = int((sent_ans['AISent_clean']==level).sum())
    pct = cnt/len(sent_ans)*100
    bar = '█' * int(pct/2)
    print(f'  {level:<20} {cnt:>7,}  {pct:>5.1f}%  {bar}')
    p2_sent_rows.append({'Sentiment_Level':level,'Count':cnt,'Percent':round(pct,1)})

# Ethics gap summary
p2_gap_df = pd.DataFrame([
    {'Dimension':'Positive Sentiment','Rate':round(pos_sent,1),'n':len(sent_ans),'Mean_Score':round(mean_sent,3)},
    {'Dimension':'Positive Trust',    'Rate':round(pos_trust,1),'n':len(trust_ans),'Mean_Score':round(mean_trust,3)},
    {'Dimension':'Ethics Gap (pp)',   'Rate':round(gap,1),'n':0,'Mean_Score':round(mean_sent-mean_trust,3)}
])
p2_sent_df = pd.DataFrame(p2_sent_rows)
print('\n[P2 ethics gap and sentiment distribution saved for export]')

## Cell 9 — P2: ToolGroup distrust analysis (Paradox 2 Proof B)

In [ ]:
# Familiarity breeds scepticism: more tools = MORE distrust

# Only rows where BOTH ToolGroup and AIAcc_clean are answered
tool_trust = df[
    (df['ToolGroup'] != 'No Response') &
    (df['AIAcc_clean'] != 'No Response')
].copy()

print('Any Distrust Rate by ToolGroup:')
print('(Expected: 41.4% → 41.4% → 47.5%)')
print()

TOOL_ORDER = ['1 Tool', '2-3 Tools', '4+ Tools']
p2_tool_rows = []
for grp in TOOL_ORDER:
    g   = tool_trust[tool_trust['ToolGroup'] == grp]
    n   = len(g)
    ad  = g['AIAcc_clean'].isin(['Somewhat distrust','Highly distrust']).mean() * 100
    ht  = (g['AIAcc_clean'] == 'Highly trust').mean() * 100
    avg = g['AIAcc_score'].mean()
    bar = '█' * int(ad / 2)
    print(f'  {grp:<12}  n={n:>6,}  Any Distrust={ad:>5.1f}%  High Trust={ht:>4.1f}%  {bar}')
    p2_tool_rows.append({
        'ToolGroup': grp, 'n': n,
        'Any_Distrust_Pct': round(ad,1),
        'High_Trust_Pct': round(ht,1),
        'Avg_Trust_Score': round(avg,3)
    })

print()
distrust_vals = [r['Any_Distrust_Pct'] for r in p2_tool_rows]
print(f'Direction: {distrust_vals[0]} → {distrust_vals[1]} → {distrust_vals[2]}')
print(f'Rise from 1-Tool to 4+Tools: {distrust_vals[2]-distrust_vals[0]:.1f} pp')
print('Conclusion: MORE tools used = MORE distrust. Familiarity breeds scepticism.')

p2_tool_df = pd.DataFrame(p2_tool_rows)
print('\n[P2 ToolGroup distrust table saved for export]')

## Cell 10 — P2: AIFrustration breakdown

In [ ]:
# Frustration counts — explains WHY distrust grows with exposure

frust_cols = {
    'Frust_AlmostRight'      : '"Almost right, needs changes every time"',
    'Frust_Debugging'        : '"Debugging AI code is more time-consuming"',
    'Frust_LessConfident'    : '"Makes me less confident in own skills"',
    'Frust_HardToUnderstand' : '"Hard to understand why code worked"',
}

print('AIFrustration breakdown (multi-select — overlaps allowed):')
print(f'  Total respondents       : {len(df):,}')
print(f'  Answered AIFrustration  : {df["AIFrustration"].notna().sum():,}  ({df["AIFrustration"].notna().mean()*100:.1f}%)')
print()
print(f'  {"Frustration":<48} {"Count":>8} {"% of total":>11}')
print('  ' + '-'*70)

p2_frust_rows = []
for col, label in frust_cols.items():
    if col in df.columns:
        cnt = int(df[col].sum())
        pct = cnt / len(df) * 100
        bar = '█' * int(pct / 2)
        print(f'  {label:<48} {cnt:>8,} {pct:>10.1f}%  {bar}')
        p2_frust_rows.append({'Frustration': label, 'Count': cnt, 'Pct_of_Total': round(pct,1)})

print()
print('The top frustration "Almost right, needs changes" explains the ethics gap:')
print('Developers enjoy the speed boost (positive sentiment) but spend extra time')
print('fixing outputs — which builds distrust rather than trust over time.')

p2_frust_df = pd.DataFrame(p2_frust_rows)
print('\n[P2 frustration data saved for export]')

---
# PART C — Paradox 3: Utility-Threat Tension
**Variables used:** AIThreat_clean, NewRole, JobSat, UsesAI, ExperienceBand

## Cell 11 — P3: Job threat distribution (current state)

In [ ]:
# Core Paradox 3 chart — donut distribution

threat_ans = df[df['AIThreat_clean'] != 'No Response'].copy()
n_threat   = len(threat_ans)

print('=' * 55)
print('  PARADOX 3 — JOB THREAT ANALYSIS')
print('=' * 55)
print(f'\nRespondents who answered AIThreat: {n_threat:,}')
print(f'No Response (skipped):             {(df["AIThreat_clean"]=="No Response").sum():,}')
print()

THREAT_ORDER = ['No', "I'm not sure", 'Yes']
p3_threat_rows = []
for val in THREAT_ORDER:
    cnt = int((threat_ans['AIThreat_clean'] == val).sum())
    pct = cnt / n_threat * 100
    bar = '█' * int(pct / 2)
    label = {'No':'Not Threatened',"I'm not sure":'Uncertain','Yes':'Threatened'}[val]
    print(f'  {label:<18} {cnt:>7,}  {pct:>5.1f}%  {bar}')
    p3_threat_rows.append({'Response':val,'Label':label,'Count':cnt,'Percent':round(pct,1)})

not_conf = threat_ans['AIThreat_clean'].isin(["I'm not sure",'Yes']).sum()
not_conf_pct = not_conf / n_threat * 100

print(f'\n  Not confident job is safe  = {not_conf:,}  ({not_conf_pct:.1f}%) [Yes + Not Sure]')
print(f'  Paradox 3 headline         : 36.2% are NOT confident their job is safe')
print(f'  Declining trend            : 70% (2023) → 68% (2024) → 63.8% (2025)')

p3_threat_df = pd.DataFrame(p3_threat_rows)
print('\n[P3 threat distribution saved for export]')

## Cell 12 — P3: 3-year trend data (manually entered — used in Power BI)

In [ ]:
# 2023 and 2024 figures are from Stack Overflow published reports
# 2025 figures are from your dataset (calculated above)

trend_data = {
    'Year': [2023, 2024, 2025],
    'Not_Threatened': [0.70, 0.68, 0.638],
    'Threatened':     [0.12, 0.13, 0.149],
    'Not_Sure':       [0.18, 0.19, 0.213]
}
p3_trend_df = pd.DataFrame(trend_data)

print('3-year trend data (ready for Power BI Enter Data):')
print()
print(f'{"Year":<8} {"Not Threatened":>15} {"Threatened":>12} {"Not Sure":>10}')
print('-' * 48)
for _, row in p3_trend_df.iterrows():
    nt_bar = '█' * int(row['Not_Threatened'] * 20)
    t_bar  = '█' * int(row['Threatened']     * 20)
    print(f'{int(row["Year"]):<8} {row["Not_Threatened"]*100:>14.1f}%  {row["Threatened"]*100:>11.1f}%  {row["Not_Sure"]*100:>9.1f}%')

decline = (trend_data['Not_Threatened'][0] - trend_data['Not_Threatened'][2]) * 100
rise    = (trend_data['Threatened'][2]     - trend_data['Threatened'][0])     * 100
print(f'\nNot Threatened fell : {decline:.1f} pp over 2 years')
print(f'Threatened rose     : {rise:.1f} pp over 2 years')

# Statistical test (2025 data vs historical expectation)
# Test independence of year and threat response
print('\nChi-square test (2025 threat distribution vs historical baseline):')
observed = [22576, 7553, 5273]  # No, Not Sure, Yes
expected_pcts = [0.70, 0.18, 0.12]  # 2023 baseline
total = sum(observed)
expected_counts = [p * total for p in expected_pcts]
chi2, p_val = stats.chisquare(observed, f_exp=expected_counts)
print(f'  χ² = {chi2:,.2f}')
print(f'  p  = {p_val:.6f}  (p < 0.0001 — highly significant)')
print('  Conclusion: decline from 70% to 63.8% is statistically significant')

print('\n[P3 trend data saved for export]')

## Cell 13 — P3: Threat by ExperienceBand (opposite of Paradox 1 expertise effect)

In [ ]:
# In P1: trust FALLS with experience (experienced most sceptical)
# In P3: threat FALLS with experience (experienced least threatened)
# This cross-paradox comparison is one of the strongest insights

print('Threat rate by ExperienceBand:')
print('(Compare with P1 trust rates — they move in OPPOSITE directions)')
print()
print(f'{"Band":<30} {"n":>7} {"Threatened %":>14} {"P1 High Trust %":>17}')
print('-' * 72)

p1_trust_ref = {
    'Early Career (0-2 yrs)': 10.0,
    'Mid-Early (3-5 yrs)': 5.4,
    'Mid (6-10 yrs)': 2.8,
    'Experienced (10+ yrs)': 2.3
}

p3_exp_rows = []
for band in BAND_ORDER:
    g   = threat_ans[threat_ans['ExperienceBand'] == band]
    n   = len(g)
    thr = (g['AIThreat_clean'] == 'Yes').mean() * 100
    ht  = p1_trust_ref.get(band, 0)
    arrow = '← MOST THREATENED' if band == 'Early Career (0-2 yrs)' else '← LEAST THREATENED' if band == 'Experienced (10+ yrs)' else ''
    print(f'{band:<30} {n:>7,} {thr:>13.1f}% {ht:>16.1f}%  {arrow}')
    p3_exp_rows.append({
        'ExperienceBand': band, 'n': n,
        'Threatened_Pct': round(thr,1),
        'P1_HighTrust_Pct': ht
    })

print()
print('KEY INSIGHT: Paradox 1 and Paradox 3 move in OPPOSITE directions:')
print('  Experienced developers: LOWEST trust (2.3%) but LEAST threatened (13.6%)')
print('  Early Career developers: HIGHEST trust (10.0%) but MOST threatened (21.2%)')
print()
print('Explanation: Experienced devs know AI cannot easily replace deep expertise.')
print('Junior devs trust AI more (less experience to detect errors) but fear')
print('displacement more (their skills are more easily replicable by AI).')

p3_exp_df = pd.DataFrame(p3_exp_rows)
print('\n[P3 threat by experience saved for export]')

## Cell 14 — P3: NewRole career disruption analysis

In [ ]:
# NewRole: have you considered or made a career change because of AI?

nr_ans = df[df['NewRole'].notna()].copy()
n_nr   = len(nr_ans)

print(f'Career change analysis (n={n_nr:,} who answered NewRole):')
print()

p3_nr_rows = []
for val, cnt in nr_ans['NewRole'].value_counts().items():
    pct = cnt / n_nr * 100
    short = str(val)[:60]
    action = 'STABLE' if 'neither' in str(val).lower() else \
             'CONCERN' if 'somewhat' in str(val).lower() else \
             'HIGH CONCERN' if 'strongly' in str(val).lower() else \
             'DISRUPTED' if 'voluntary' in str(val).lower() else 'FORCED'
    bar = '█' * int(pct / 2)
    print(f'  [{action:<12}] {cnt:>6,}  {pct:>5.1f}%  {bar}')
    print(f'    "{short}"')
    p3_nr_rows.append({'NewRole': str(val)[:80], 'Count': cnt, 'Percent': round(pct,1), 'Category': action})

# Calculate: at least considered
at_least = nr_ans[~nr_ans['NewRole'].str.lower().str.contains('neither', na=False)]
at_pct   = len(at_least) / n_nr * 100

print(f'\n  HEADLINE: {at_pct:.1f}% have at least considered a career change')
print(f'  Of these, {len(nr_ans[nr_ans["NewRole"].str.contains("involuntarily", na=False)]):,} transitioned INVOLUNTARILY')

p3_nr_df = pd.DataFrame(p3_nr_rows)
print('\n[P3 NewRole data saved for export]')

## Cell 15 — P3: JobSat comparison (AI users vs non-users)

In [ ]:
# Job satisfaction by AI adoption — supporting evidence for Paradox 3

jobsat_data = df[df['JobSat'].notna()].copy()

ai_grp  = jobsat_data[jobsat_data['UsesAI'] == 'Uses AI']['JobSat']
non_grp = jobsat_data[jobsat_data['UsesAI'] == 'Does Not Use AI']['JobSat']

print('Job Satisfaction by AI Adoption Status:')
print(f'  (Only employed respondents who answered JobSat: n={len(jobsat_data):,})')
print()
print(f'  AI users       : n={len(ai_grp):,}  mean={ai_grp.mean():.2f}  median={ai_grp.median():.1f}')
print(f'  Non-AI users   : n={len(non_grp):,}  mean={non_grp.mean():.2f}  median={non_grp.median():.1f}')
print(f'  Difference     : {ai_grp.mean()-non_grp.mean():.2f} pts  (AI users slightly lower)')

# t-test
t_stat, t_p = stats.ttest_ind(ai_grp, non_grp)
print(f'\n  t-test: t={t_stat:.3f}, p={t_p:.4f}')
print(f'  Statistical significance: {"Yes (p<0.05)" if t_p < 0.05 else "No (p>0.05)"}')
print()
print('  Interpretation: AI users are marginally less satisfied.')
print('  The utility-threat tension slightly erodes job satisfaction.')
print('  This is supporting evidence for Paradox 3, not the core proof.')

p3_jobsat_df = pd.DataFrame([
    {'Group':'Uses AI', 'n':len(ai_grp), 'Mean_JobSat':round(ai_grp.mean(),2), 'Median_JobSat':ai_grp.median()},
    {'Group':'Does Not Use AI', 'n':len(non_grp), 'Mean_JobSat':round(non_grp.mean(),2), 'Median_JobSat':non_grp.median()},
])
print('\n[P3 JobSat comparison saved for export]')

---
# PART D — Create Power BI Optimised Dataset

## Cell 16 — Build trust_paradox_powerbi.csv (37 columns for Power BI)

In [ ]:
# Keep only the 37 most useful columns — removes open-text, near-empty columns
# and raw technology preference multi-selects (languages, databases, frameworks)
# Power BI loads faster and the data model is cleaner

PBI_COLS = [
    # Identity
    'ResponseId',
    # Demographics
    'Age', 'EdLevel', 'Employment', 'WorkExp', 'Country', 'DevType',
    'OrgSize', 'RemoteWork', 'ICorPM', 'Industry',
    # Experience (raw + derived)
    'YearsCode', 'YearsCode_num', 'ExperienceBand',
    # AI Adoption (raw + derived)
    'AISelect', 'UsesAI', 'LearnCodeAI',
    'ToolCountWork', 'ToolCountPersonal', 'ToolGroup',
    'AIAgents', 'AIAgentChange', 'AIModelsChoice',
    # AI Trust — Paradox 1 (raw + derived)
    'AIAcc', 'AIAcc_clean', 'AIAcc_score',
    # AI Sentiment — Paradox 2 (raw + derived)
    'AISent', 'AISent_clean', 'AISent_score',
    # AI Quality
    'AIComplex', 'AIFrustration',
    # Job Threat — Paradox 3 (raw + derived)
    'AIThreat', 'AIThreat_clean',
    # Career / Wellbeing
    'JobSat', 'JobSat_clean', 'NewRole',
    # SO usage
    'SOAccount', 'SOVisitFreq',
]

# Only keep columns that actually exist in the dataframe
pbi_cols_present = [c for c in PBI_COLS if c in df.columns]
df_pbi = df[pbi_cols_present].copy()

# Fill string NaN with empty string for cleaner loading in Power BI
for col in df_pbi.columns:
    if df_pbi[col].dtype == object:
        df_pbi[col] = df_pbi[col].fillna('')

print(f'Power BI dataset:')
print(f'  Rows    : {len(df_pbi):,}')
print(f'  Columns : {len(df_pbi.columns)}')
print(f'  Removed : {len(df.columns) - len(df_pbi.columns)} columns (open-text, tech multi-selects, near-empty)')
print()
print('Columns included:')
for i, col in enumerate(df_pbi.columns, 1):
    tag = '(DERIVED)' if col in ['AIAcc_clean','AIAcc_score','AISent_clean','AISent_score',
                                   'AIThreat_clean','JobSat_clean','UsesAI','ExperienceBand',
                                   'YearsCode_num','ToolGroup'] else ''
    print(f'  {i:>2}. {col:<30} {tag}')

# Save
PBI_PATH = '/content/trust_paradox_powerbi.csv'
df_pbi.to_csv(PBI_PATH, index=False)
size_mb = os.path.getsize(PBI_PATH) / 1024 / 1024
print(f'\nSaved: {PBI_PATH}  ({size_mb:.1f} MB)')

## Cell 17 — Save all 4 analysis summary CSVs

In [ ]:
# Page 1 — Overview summary
p1_overview = pd.DataFrame([
    {'Metric':'Adoption Rate %',     'Value':round(adopt_rate,1)},
    {'Metric':'High Trust Rate %',   'Value':round(ht_pct,1)},
    {'Metric':'Any Distrust Rate %', 'Value':round(ad_pct,1)},
    {'Metric':'Not Threatened %',    'Value':63.8},
    {'Metric':'Avg Trust Score',     'Value':round(mean_sc,3)},
    {'Metric':'Paradox Severity pts','Value':round(adopt_rate-ht_pct,1)},
    {'Metric':'Total Respondents',   'Value':len(df)},
    {'Metric':'Trust Answered',      'Value':n_trust},
])

PATHS = {
    '/content/p1_overview_kpis.csv'      : p1_overview,
    '/content/p1_trust_distribution.csv' : p1_dist_df,
    '/content/p1_crosstab_usesai.csv'    : p1_crosstab_df,
    '/content/p1_expertise_effect.csv'   : p1_expertise_df,
    '/content/p1_country_trust.csv'      : p1_country_df,
    '/content/p2_ethics_gap.csv'         : p2_gap_df,
    '/content/p2_sentiment_dist.csv'     : p2_sent_df,
    '/content/p2_toolgroup_distrust.csv' : p2_tool_df,
    '/content/p2_frustrations.csv'       : p2_frust_df,
    '/content/p3_threat_dist.csv'        : p3_threat_df,
    '/content/p3_trend_3year.csv'        : p3_trend_df,
    '/content/p3_threat_by_exp.csv'      : p3_exp_df,
    '/content/p3_newrole.csv'            : p3_nr_df,
    '/content/p3_jobsat_comparison.csv'  : p3_jobsat_df,
}

print('Saving analysis CSVs...')
for path, data in PATHS.items():
    data.to_csv(path, index=False)
    fname = path.split('/')[-1]
    print(f'  ✓ {fname:<45} {len(data)} rows')

print(f'\nTotal files saved: {len(PATHS) + 1}  (+1 for power BI dataset)')

## Cell 18 — Download all files to your computer

In [ ]:
from google.colab import files

ALL_FILES = [PBI_PATH] + list(PATHS.keys())

print('Downloading all files...')
print('(Your browser will trigger one download per file)')
print()

for path in ALL_FILES:
    fname = path.split('/')[-1]
    print(f'  Downloading: {fname}')
    files.download(path)

print()
print('=' * 55)
print('  OBJECTIVE 2 COMPLETE')
print('=' * 55)
print()
print('  Files produced:')
print('  trust_paradox_powerbi.csv  → load into Power BI')
print('  p1_*.csv                   → Page 1 & 2 analysis')
print('  p2_*.csv                   → Page 3 analysis')
print('  p3_*.csv                   → Page 4 analysis')
print()
print('  Next: open Power BI Desktop → Get Data → Text/CSV')
print('  Select trust_paradox_powerbi.csv and build 4 pages.')